# 📝 과제 01: 브랜드 취향을 기억하는 홍보 이미지 에이전트를 만듭니다

여러 브랜드의 홍보 이미지를 만드는 콘텐츠 담당자입니다. 같은 이미지 요청에 브랜드별 화풍·색·구도를 적용하고, 다음 제작에서도 저장한 선호를 재사용합니다.

**연결 교안:** 교안 01 Store로 사용자 기억 공유하기

## 이번 과제에서 만들 에이전트

- **입력:** 브랜드 ID와 공통 이미지 요청
- **처리:** recall_brand가 브랜드 배경·화풍·관련 기억을 읽고 generate_image가 생성 프롬프트와 PNG 파일을 만듭니다.
- **결과:** 브랜드 A·B의 홍보 이미지와 실제 생성 프롬프트. 새 제작과 선호 변경 뒤의 결과도 비교합니다.

<img src="images/assignment01_overview.png" width="1050" alt="과제 01 에이전트의 노드·기억 저장소·State 흐름">

**진행:** 브랜드 기억 저장 → key 조회·의미 검색 → 이미지 생성 → 두 브랜드 비교 → 선호 변경 → 파일 재연결

자료는 실무 상황을 단순화한 **학습용 가상 데이터**입니다. 문항을 순서대로 작성하세요.

**풀이 방법:** ‘코드 골격 완성’에서는 `...` 부분만 작성합니다. 긴 프롬프트·출력·파일 처리·반복 실행 틀은 제공하므로 그대로 사용하세요. 요구사항은 완성된 코드의 동작을 설명하며, 이미 제공된 줄은 다시 작성하지 않습니다. 빈칸을 채우기 전에는 해당 작성 셀과 뒤의 자가평가를 실행하지 마세요.

**전체 코드 작성:** 3번 CRUD, 5번 State·사용자 context는 짧은 코드 전체를 직접 작성합니다. 나머지 문항은 핵심 연산을 채우며 여러 개념을 연결합니다.

**공통 작성 규칙:** 노드는 입력 State와 그 안의 목록을 직접 바꾸지 않고 갱신할 필드만 반환합니다. 호출 입력은 위치 인자와 키워드 인자 모두 사용할 수 있습니다. messages 필드를 사용하는 과제에서는 새 메시지를 add_messages로 누적합니다.

**자가평가:** 작성 셀 다음의 assert 셀을 실행하세요. 노드 평가는 고정 응답으로 입출력을 검사하며 GPT·임베딩을 추가 호출하지 않습니다. 실행 문항은 앞에서 만든 결과를 읽습니다. 생성 문장의 의미와 추출한 기억의 적절성은 원자료와 직접 대조하세요.

**실행 안내:** 지정된 실행은 이미지 생성 4회입니다(9번 두 브랜드, 10번 새 작업, 11번 변경 후 작업). 저장·의미 검색에는 임베딩 호출이 포함됩니다. 자가평가 셀은 추가 이미지·임베딩 호출을 하지 않습니다. 이미지 품질은 요청과 선호를 기준으로 직접 확인합니다.

## 준비: 모델과 업무 자료를 읽습니다


In [ ]:
import os
import json
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, Markdown, display

# 현재 작업 폴더부터 상위 폴더로 .env를 찾아 환경 설정 읽기
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)


In [ ]:
# [제공 코드] 교안에서 사용한 메시지·그래프·평가용 도구를 준비합니다.
import sqlite3
from dataclasses import dataclass, is_dataclass
from typing import Annotated, get_type_hints, get_origin, get_args
from typing_extensions import TypedDict, is_typeddict
from types import SimpleNamespace
from unittest.mock import Mock, patch
from uuid import uuid4
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage, RemoveMessage
from langchain_core.messages.utils import count_tokens_approximately, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.runtime import Runtime
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore


In [ ]:
# [제공 코드] 브랜드별 원자료와 공통 요청을 읽습니다.
brands = json.loads((data_dir / "assignment01_brands.json").read_text(encoding="utf-8"))
common_request = "신학기 책상 꾸미기 홍보 이미지를 만들어 주세요. 무라벨 텀블러, 펼친 노트, 작은 화분을 책상 위에 배치하세요. 글자·로고·워터마크는 넣지 마세요."
for user_id, brand in brands.items():
    print(user_id, brand)
print("공통 요청:", common_request)


In [ ]:
# [제공 코드] 기억과 제작 State는 교안처럼 별도 파일에 저장합니다.
from langchain_openai import OpenAIEmbeddings
store_db = output_dir / "assignment01_store.sqlite"
store_connection = sqlite3.connect(str(store_db), check_same_thread=False, isolation_level=None)
store_connection.execute("PRAGMA foreign_keys = ON")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768)
store = SqliteStore(store_connection, index={"embed": embeddings, "dims":768})
store.setup()
connection = sqlite3.connect(str(output_dir / "assignment01_jobs.sqlite"), check_same_thread=False)
checkpointer = SqliteSaver(connection)


## 1. 브랜드별 namespace에 기억을 저장합니다

**배경:** 브랜드의 장기 선호와 이번 제작 요청을 구분합니다.

**작성:** 코드 골격 완성

**제공:** 브랜드 반복문·kind·value를 제공합니다. namespace와 저장 시 인덱스 범위를 완성하세요.

**요구사항:**
- 브랜드마다 `namespace`를 용도 `brand_assets`와 해당 사용자 ID의 튜플로 만드세요.
- `profile`, `visual_style`, `layout_note`, `detail_note`를 같은 이름의 key로 저장하세요. value는 `text`에 원자료를, `kind`에 profile은 `profile`, 나머지는 `preference`를 담습니다.
- profile·visual_style은 key로 조회하므로 index=False로, 두 note는 의미 검색하므로 text 필드만 임베딩 대상으로 저장하세요.

<details><summary>힌트</summary>

```text
key로 항상 읽을 항목과 질문의 의미로 찾을 항목을 나눠 저장합니다.
```

</details>


In [ ]:
# [작성] 사용자별 저장 범위와 의미 검색할 필드를 정하세요.
for user_id, brand in brands.items():
    namespace = ...
    for key, text in brand.items():
        kind = "profile" if key == "profile" else "preference"
        value = {"text": text, "kind": kind}
        # note만 의미 검색에 사용하고 나머지는 key로 조회합니다.
        index_fields = ...
        store.put(namespace, key, value, index=index_fields)


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_1():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    for user_id, brand in brands.items():
        for key, text in brand.items():
            item = store.get(("brand_assets", user_id), key)
            assert item is not None, "해당 브랜드 namespace와 key에 기억을 저장하세요."
            assert item.value["text"] == text, "value의 text에 해당 원자료를 넣으세요."
            assert item.value["kind"] == ("profile" if key == "profile" else "preference"), "kind를 요구사항대로 구분하세요."
    print("1번 자가평가 통과")

grade_question_1()


## 2. key로 브랜드의 화풍을 읽습니다

**배경:** 항상 사용할 화풍·색·구도는 검색 순위에 맡기지 않습니다.

**작성:** 코드 골격 완성

**제공:** 조회·텍스트 추출용 변수와 출력 코드를 제공합니다.

**요구사항:**
- `style_item`에 studio-a의 visual_style 항목을 조회하고, value의 text를 문자열 `style_text`에 담아 출력하세요.

<details><summary>힌트</summary>

```text
get은 value 자체가 아니라 기억 항목을 반환합니다.
```

</details>


In [ ]:
# [작성] key로 읽은 항목에서 실제 화풍 문자열을 꺼내세요.
style_item = ...
style_text = ...
print("저장된 화풍:", style_text)


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_2():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert style_item.key == "visual_style", "visual_style key를 조회하세요."
    assert style_text == brands["studio-a"]["visual_style"], "value의 text를 꺼내세요."
    print("2번 자가평가 통과")

grade_question_2()


## 3. 테스트 기억을 수정하고 삭제합니다

**배경:** 검토 중 임시로 저장한 지시를 장기 선호에 남기지 않습니다.

**작성:** 전체 코드 작성

**제공:** 테스트 영역은 (brand_assets, sandbox)로 지정합니다.

**요구사항:**
- 테스트 영역의 `test_style`에 text가 `진한 배경을 사용합니다.`인 value를 저장하세요.
- 같은 namespace·key에 text가 `밝은 배경을 사용합니다.`인 value로 교체하고, 수정된 text를 `updated_text`에 담으세요.
- 그 key를 삭제하고 조회 결과를 `deleted_item`에 담으세요. 브랜드 A·B의 기억은 그대로 유지합니다.

<details><summary>힌트</summary>

```text
수정은 같은 key로 다시 저장하고, 삭제 후 get의 반환값을 확인합니다.
```

</details>


In [ ]:
# [작성] 위 요구사항에 맞춰 코드를 작성하세요.


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_3():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert updated_text == "밝은 배경을 사용합니다.", "같은 key에 새 값을 저장하세요."
    assert deleted_item is None, "삭제 후 get 결과를 확인하세요."
    print("3번 자가평가 통과")

grade_question_3()


In [ ]:
# [제공 코드] 문항 4의 검색 질문입니다.
search_question = "상품 배치와 여백, 재질은 어떻게 표현하면 좋을까요?"


## 4. 의미 검색으로 구도 참고를 찾습니다

**배경:** 기억의 key를 모르는 질문에도 관련 제작 선호를 찾습니다.

**작성:** 코드 골격 완성

**제공:** search_question과 검색 결과 출력 반복문을 제공합니다.

**요구사항:**
- `search_question`을 검색어로 studio-a의 namespace에서 search를 사용하세요. filter는 kind가 preference인 항목, limit은 2로 지정합니다.
- 반환 목록을 `style_hits`에 담고 각 항목의 key·value의 text·score를 출력하세요. 저장한 두 note 중 한 건 이상을 찾을 수 있어야 합니다.
- 검색 순위와 score는 실행마다 다를 수 있습니다. 직접 출력에서 질문과 관련된 구도·질감 설명인지 확인하세요.

<details><summary>힌트</summary>

```text
namespace는 브랜드 범위, filter는 value의 조건, query는 의미 검색 질문입니다.
```

</details>


In [ ]:
# [작성] 사용자 범위·query·filter·limit을 지정해 검색하세요.
style_hits = ...
# [제공] 검색 순위와 관련성을 원문으로 확인합니다.
for item in style_hits:
    print(item.key, item.value["text"], item.score)


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_4():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert isinstance(style_hits, list) and 1 <= len(style_hits) <= 2, "limit 2의 검색 목록을 담으세요."
    assert all(item.namespace == ("brand_assets", "studio-a") and item.value["kind"] == "preference" for item in style_hits), "브랜드 범위와 filter를 확인하세요."
    print("4번 자가평가 통과")

grade_question_4()


## 5. 제작 State와 사용자 context를 정의합니다

**배경:** 현재 제작 결과는 State, 기억의 소유자 ID는 실행 context로 전달합니다.

**작성:** 전체 코드 작성

**제공:** TypedDict·dataclass를 제공합니다.

**요구사항:**
- `ImageState(TypedDict)`에 request·profile·preferences·generation_prompt·image_path를 str로, recalled_memories를 list[str]로 선언하세요.
- `UserContext`를 dataclass로 정의하고 user_id: str을 선언하세요.
- `image_input`에는 request에 common_request만 담으세요. 나머지는 노드가 반환합니다.

<details><summary>힌트</summary>

```text
큰 이미지 데이터는 State에 넣지 않고 파일 경로로 보관합니다.
```

</details>


In [ ]:
# [작성] 위 요구사항에 맞춰 코드를 작성하세요.


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_5():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert is_typeddict(ImageState), "ImageState는 TypedDict로 선언하세요."
    assert set(ImageState.__annotations__) == {"request","profile","preferences","recalled_memories","generation_prompt","image_path"}, "요구한 State 필드를 확인하세요."
    state_types = get_type_hints(ImageState,include_extras=True)
    assert all(state_types[field] is str for field in ["request","profile","preferences","generation_prompt","image_path"]), "문자열 State 필드는 str로 선언하세요."
    assert get_origin(state_types["recalled_memories"]) is list and get_args(state_types["recalled_memories"]) == (str,), "recalled_memories는 문자열 목록입니다."
    assert is_dataclass(UserContext) and get_type_hints(UserContext)["user_id"] is str, "user_id: str을 가진 dataclass로 선언하세요."
    assert image_input == {"request":common_request}, "초기 입력에는 이번 요청을 넣으세요."
    assert UserContext(user_id="studio-a").user_id == "studio-a", "context에 사용자 ID를 담으세요."
    print("5번 자가평가 통과")

grade_question_5()


## 6. 현재 브랜드의 기억을 읽는 노드를 만듭니다

**배경:** 다른 브랜드의 기억이 생성 프롬프트에 섞이지 않게 합니다.

**작성:** 코드 골격 완성

**제공:** 함수·반환 딕셔너리·기억 없음 처리 틀을 제공합니다. 조회 범위와 API 호출을 채우세요.

**요구사항:**
- `recall_brand`는 runtime.context.user_id로 brand_assets namespace를 정합니다.
- profile·visual_style을 get으로 조회합니다. 있으면 각각 value의 text, 없으면 빈 문자열을 profile·preferences로 반환합니다.
- State의 request를 query로 search하고 kind=preference, limit=2를 지정합니다. 검색 항목의 text 목록을 recalled_memories로 반환하세요.
- 입력 State는 직접 수정하지 않습니다.

<details><summary>힌트</summary>

```text
항상 필요한 배경·화풍은 get으로, 추가 참고는 search로 읽습니다.
```

</details>


In [ ]:
# [작성] 현재 브랜드의 조회 범위와 기억을 완성하세요.
def recall_brand(state: ImageState, runtime: Runtime[UserContext]):
    """현재 브랜드의 배경·화풍·관련 제작 참고를 반환합니다."""
    namespace = ...
    profile = ...
    style = ...
    hits = ...
    return {
        "profile": profile.value["text"] if profile else "",
        "preferences": style.value["text"] if style else "",
        "recalled_memories": [item.value["text"] for item in hits],
    }


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_6():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    fake_store = Mock()
    # 호출 순서가 아니라 key에 맞는 자료를 돌려줍니다.
    def sample_get(namespace, key):
        """평가용 기억을 key에 따라 반환합니다."""
        return {"profile":SimpleNamespace(value={"text":"배경"}),
            "visual_style":SimpleNamespace(value={"text":"화풍"})}[key]
    fake_store.get.side_effect = sample_get
    fake_store.search.return_value = [SimpleNamespace(value={"text":"제작 참고"})]
    node_input = {"request":"공통 요청"}
    checked = recall_brand(node_input, Runtime(context=UserContext(user_id="sample-brand"), store=fake_store))
    assert checked == {"profile":"배경","preferences":"화풍","recalled_memories":["제작 참고"]}, "조회 결과를 요구한 필드로 반환하세요."
    assert node_input == {"request":"공통 요청"}, "입력 State를 직접 바꾸지 마세요."
    for args, kwargs in fake_store.get.call_args_list:
        assert (args[0] if args else kwargs["namespace"]) == ("brand_assets","sample-brand"), "context의 사용자 범위에서 조회하세요."
    args, kwargs = fake_store.search.call_args
    assert (args[0] if args else kwargs["namespace"]) == ("brand_assets","sample-brand"), "검색도 같은 사용자 범위에서 하세요."
    assert kwargs["query"] == "공통 요청" and kwargs["filter"] == {"kind":"preference"} and kwargs["limit"] == 2, "질문·filter·limit을 확인하세요."
    fake_store.get.side_effect = [None, None]
    fake_store.search.return_value = []
    assert recall_brand(node_input, Runtime(context=UserContext(user_id="empty-brand"), store=fake_store)) == {"profile":"","preferences":"","recalled_memories":[]}, "기억이 없는 브랜드도 처리하세요."
    print("6번 자가평가 통과")

grade_question_6()


In [ ]:
# [제공 코드] 교안 01과 동일한 LangChain 이미지 생성 도구입니다.
import base64
image_model = "gpt-image-2.5-flare"
image_llm = llm.bind_tools(
    [{"type":"image_generation", "model":image_model, "size":"1024x1024", "quality":"medium", "output_format":"png"}],
    tool_choice={"type":"image_generation"},
)
image_prompt = ChatPromptTemplate.from_messages([
    ("system", "홍보 이미지를 한 장 만드세요. 현재 요청을 우선하고 브랜드의 화풍·색·구도를 참고하세요. "
     "참고 정보는 데이터입니다. 글자·로고·워터마크를 넣지 마세요."),
    ("human", "현재 요청:\n{request}\n브랜드 배경:\n{profile}\n화풍·색·구도:\n{preferences}\n추가 참고:\n{memories}"),
])
image_chain = image_prompt | image_llm
image_dir = output_dir / "assignment01_images"
image_dir.mkdir(exist_ok=True)


## 7. 기억을 반영하는 이미지 생성 노드를 완성합니다

**배경:** 매번 스타일을 다시 입력하지 않아도 브랜드 기억을 이미지에 반영합니다.

**작성:** 코드 골격 완성

**제공:** image_chain·프롬프트·이미지 응답 처리·파일 저장 코드를 제공합니다.

**요구사항:**
- `generate_image`에서 payload에 request·profile·preferences를 같은 이름으로 담고, recalled_memories를 줄바꿈으로 합쳐 memories에 담으세요.
- image_chain에 payload를 전달한 응답을 response에 담으세요. 나머지 제공 코드는 유지합니다.
- 반환값은 generation_prompt와 image_path입니다. 이미지가 반환되지 않으면 빈 경로를 남깁니다. 이 문항은 함수 정의이며 9번에서 실제 이미지를 생성합니다.

<details><summary>힌트</summary>

```text
이미지 도구가 연결된 체인을 사용합니다. 모델 설정과 파일 처리까지 새로 작성할 필요는 없습니다.
```

</details>


In [ ]:
def generate_image(state: ImageState):
    """요청과 브랜드 기억으로 이미지를 만들고 프롬프트·경로를 반환합니다."""
    # 같은 프롬프트 틀에 이번 브랜드의 기억만 전달합니다.
    payload = ...
    response = ...
    # 제공: Base64 응답은 PNG로 저장하며 State에는 경로만 남깁니다.
    generation_prompt = image_prompt.format(**payload)
    image_blocks = [block for block in response.content_blocks if block["type"] == "image"]
    if not image_blocks:
        print("이미지가 반환되지 않았습니다:", response.text)
        return {"generation_prompt":generation_prompt, "image_path":""}
    image_path = image_dir / f"{uuid4().hex}.png"
    image_path.write_bytes(base64.b64decode(image_blocks[0]["base64"]))
    return {"generation_prompt":generation_prompt, "image_path":str(image_path)}


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_7():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    fake_chain = Mock()
    fake_chain.invoke.return_value = SimpleNamespace(content_blocks=[], text="평가용: 이미지 없음")
    node_input = {"request":"텀블러", "profile":"생활 소품", "preferences":"수채화", "recalled_memories":["여백","나무 질감"]}
    with patch.dict(generate_image.__globals__, {"image_chain":fake_chain}):
        checked = generate_image(node_input)
    args, kwargs = fake_chain.invoke.call_args
    payload_checked = args[0] if args else kwargs["input"]
    assert all(payload_checked[key] == value for key,value in {"request":"텀블러","profile":"생활 소품","preferences":"수채화","memories":"여백\n나무 질감"}.items()), "요청과 조회한 기억을 프롬프트 입력으로 전달하세요."
    assert checked["image_path"] == "" and "수채화" in checked["generation_prompt"], "이미지 없음 처리와 생성 프롬프트 반환을 확인하세요."
    print("7번 자가평가 통과")

grade_question_7()


## 8. Store와 체크포인터를 이미지 그래프에 연결합니다

**배경:** 기억은 브랜드별로 공유하고 제작 State는 작업별로 보관합니다.

**작성:** 코드 골격 완성

**제공:** StateGraph 생성 틀과 노드 등록을 제공합니다. context_schema·엣지·저장소 연결을 완성하세요.

**요구사항:**
- image_builder를 ImageState·UserContext로 만들고 recall_brand·generate_image를 같은 이름으로 등록하세요.
- START → recall_brand → generate_image → END로 연결하세요.
- store와 checkpointer를 모두 연결해 image_graph로 컴파일하세요.

<details><summary>힌트</summary>

```text
context_schema는 사용자 정보의 형태, compile의 store는 노드가 사용할 기억 저장소입니다.
```

</details>


In [ ]:
# [작성] 실행 context의 형식과 조회→생성 순서를 연결하세요.
image_builder = StateGraph(ImageState, context_schema=...)
image_builder.add_node("recall_brand", recall_brand)
image_builder.add_node("generate_image", generate_image)
image_builder.add_edge(START, ...)
image_builder.add_edge("recall_brand", ...)
image_builder.add_edge("generate_image", END)
image_graph = image_builder.compile(store=..., checkpointer=...)


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_8():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert image_graph.store is store and image_graph.checkpointer is checkpointer, "Store와 체크포인터를 모두 연결하세요."
    assert set(image_builder.nodes) == {"recall_brand","generate_image"}, "두 노드를 등록하세요."
    assert {(START,"recall_brand"),("recall_brand","generate_image"),("generate_image",END)} <= image_builder.edges, "조회 다음 생성으로 연결하세요."
    print("8번 자가평가 통과")

grade_question_8()


## 9. 같은 요청으로 두 브랜드의 이미지를 만듭니다

**배경:** 광고 소재를 브랜드별로 만드는 작업을 자동화합니다.

**작성:** 코드 골격 완성

**제공:** 결과 딕셔너리·반복문·새 작업 ID·출력을 제공합니다. 해당 브랜드 context로 실행하는 한 줄을 작성하세요.

**요구사항:**
- image_results·job_configs를 빈 딕셔너리로 시작하세요.
- brands의 사용자 ID마다 새 thread_id를 만든 config를 job_configs[user_id]에 저장하세요.
- 같은 image_input을 실행하되 context에 해당 user_id를 전달하고 결과를 image_results[user_id]에 담으세요.
- 사용자·조회 선호·이미지 경로를 출력하세요. 총 두 번 이미지 생성을 요청합니다. 새 작업은 thread_id를 새로 만드세요.

<details><summary>힌트</summary>

```text
작업마다 config는 새로 만들고, 브랜드별 사용자 ID는 저장할 때와 같게 유지합니다.
```

</details>


In [ ]:
# [제공] 제작마다 새 대화를 시작하고 결과를 모읍니다.
image_results, job_configs = {}, {}
for user_id in brands:
    config = {"configurable": {"thread_id": str(uuid4())}}
    job_configs[user_id] = config
    # [작성] 같은 요청에 현재 브랜드의 user_id를 전달하세요.
    result = ...
    image_results[user_id] = result
    print(user_id, result["preferences"], result["image_path"])


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_9():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert set(image_results) == set(brands) and set(job_configs) == set(brands), "두 브랜드 결과와 config를 담으세요."
    assert len({config["configurable"]["thread_id"] for config in job_configs.values()}) == 2, "제작별 thread_id를 구분하세요."
    for user_id,result in image_results.items():
        assert result["request"] == common_request and result["preferences"] == brands[user_id]["visual_style"], "같은 요청과 해당 브랜드 선호를 사용하세요."
        assert result["image_path"] and Path(result["image_path"]).is_file(), "이미지가 없으면 위 응답을 확인한 뒤 다시 생성하세요."
    print("9번 자가평가 통과")

grade_question_9()


In [ ]:
# [제공 코드] 생성한 PNG를 나란히 표시합니다. API를 다시 호출하지 않습니다.
from html import escape
from IPython.display import HTML
cards = []
for user_id, result in image_results.items():
    if result["image_path"]:
        encoded = base64.b64encode(Path(result["image_path"]).read_bytes()).decode("ascii")
        image_html = f'<img src="data:image/png;base64,{encoded}" style="width:100%;height:auto">'
    else:
        image_html = "이미지가 없습니다. 생성 응답을 확인하세요."
    cards.append(f'<div style="flex:1;min-width:260px"><h4>{escape(user_id)}</h4><p>{escape(result["preferences"])}</p>{image_html}</div>')
display(HTML('<div style="display:flex;gap:20px;flex-wrap:wrap">'+"".join(cards)+'</div>'))


In [ ]:
# [제공 코드] 다음 제작의 요청입니다.
new_request = "주말 독서 홍보 이미지입니다. 책 한 권과 무라벨 머그컵을 창가에 배치하고 글자·로고는 빼 주세요."


## 10. 같은 브랜드의 새 제작에서도 기억을 씁니다

**배경:** 새 캠페인마다 화풍을 다시 입력하는 일을 줄입니다.

**작성:** 코드 골격 완성

**제공:** new_request·새 작업 config·결과 출력을 제공합니다. 같은 브랜드로 새 요청을 실행하세요.

**요구사항:**
- 새 config를 new_config에 만들고, request에 new_request만 담은 입력을 image_graph로 실행하세요.
- user_id는 studio-a로 유지하고 결과를 new_result에 담으세요. 이미지 1회를 생성합니다.
- preferences와 image_path를 출력하세요. 이전 작업의 request를 새 입력에 섞지 않습니다.

<details><summary>힌트</summary>

```text
thread_id를 바꾸면 새 State를 만들고, 같은 user_id로 Store 기억을 다시 읽습니다.
```

</details>


In [ ]:
# [제공] 새 제작은 새 State에서 시작합니다.
new_config = {"configurable": {"thread_id": str(uuid4())}}
# [작성] studio-a의 기억을 재사용하는 실행을 완성하세요.
new_result = ...
print(new_result["preferences"], new_result["image_path"])


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_10():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert new_config["configurable"]["thread_id"] not in {config["configurable"]["thread_id"] for config in job_configs.values()}, "새 제작은 새 thread_id로 시작하세요."
    assert new_result["request"] == new_request and new_result["preferences"] == brands["studio-a"]["visual_style"], "새 요청에서 A의 기억을 재사용하세요."
    assert image_graph.get_state(job_configs["studio-a"]).values["request"] == common_request, "기존 제작 State는 별도 thread에 남겨 두세요."
    assert new_result["image_path"] and Path(new_result["image_path"]).is_file(), "새 제작 이미지가 없으면 위 응답을 확인하세요."
    print("10번 자가평가 통과")

grade_question_10()


In [ ]:
# [제공 코드] 색은 유지하고 화풍만 바꾸는 요청입니다.
updated_style = "평면 종이 콜라주, 크림·연두·연분홍, 여백이 넓은 구도를 선호합니다."


## 11. 변경된 화풍을 다음 이미지에 적용합니다

**배경:** 브랜드의 리뉴얼된 스타일을 저장하고 다음 제작에 반영합니다.

**작성:** 코드 골격 완성

**제공:** updated_style·새 제작 실행·이미지 표시를 제공합니다. 기존 visual_style의 저장 값만 갱신하세요.

**요구사항:**
- studio-a의 visual_style만 updated_style로 바꾸세요. kind=preference, index=False를 유지합니다.
- 다른 key의 제작 참고는 유지합니다. 새 thread_id의 changed_config에서 같은 new_request를 생성해 changed_result에 담으세요.
- 변경한 화풍을 출력하고 새 이미지를 표시하세요. 이미지 1회를 생성합니다. B의 화풍은 유지합니다.

<details><summary>힌트</summary>

```text
변경은 A의 같은 namespace·key에 적용하며, 다음 제작에서 다시 조회합니다.
```

</details>


In [ ]:
# [작성] A의 같은 namespace·key를 updated_style로 갱신하세요.
...
# [제공] 변경한 기억을 새 작업에서 다시 읽어 이미지를 만듭니다.
changed_config = {"configurable": {"thread_id": str(uuid4())}}
changed_result = image_graph.invoke({"request": new_request}, changed_config,
    context=UserContext(user_id="studio-a"))
print("변경된 화풍:", changed_result["preferences"])
if changed_result["image_path"]:
    display(Image(filename=changed_result["image_path"]))


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_11():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert changed_result["preferences"] == updated_style, "새 작업에서 변경된 선호를 조회하세요."
    assert changed_result["image_path"] and Path(changed_result["image_path"]).is_file(), "변경 후 생성 이미지가 없으면 위 응답을 확인하세요."
    assert changed_config["configurable"]["thread_id"] != new_config["configurable"]["thread_id"], "변경 후 제작은 새 thread로 실행하세요."
    assert store.get(("brand_assets","studio-b"),"visual_style").value["text"] == brands["studio-b"]["visual_style"], "B의 기억은 그대로 유지하세요."
    print("11번 자가평가 통과")

grade_question_11()


## 12. 같은 파일을 다시 열어 기억을 읽습니다

**배경:** 노트북 연결을 닫아도 저장한 브랜드 선호는 남습니다.

**작성:** 코드 골격 완성

**제공:** 동일 파일 재연결·Store 생성·setup·출력을 제공합니다. 저장된 선호를 다시 조회하세요.

**요구사항:**
- store_connection을 닫고 같은 store_db에 새 SQLite 연결을 만들어 같은 변수에 담으세요. check_same_thread=False, isolation_level=None, foreign_keys ON 설정을 유지합니다.
- 같은 임베딩 모델·768차원의 SqliteStore를 store에 다시 연결하고 setup을 실행하세요.
- A의 visual_style을 조회한 text를 restored_style에 담으세요. 그래프를 다시 실행한다면 image_builder를 새 store로 다시 컴파일해야 합니다. 여기서는 get 조회까지만 수행합니다.

<details><summary>힌트</summary>

```text
같은 파일과 namespace·key로 기억을 읽습니다. 기존 그래프는 닫힌 Store 연결을 참조합니다.
```

</details>


In [ ]:
# [제공] 파일은 유지하고 닫힌 연결만 새로 엽니다.
store_connection.close()
store_connection = sqlite3.connect(str(store_db), check_same_thread=False, isolation_level=None)
store_connection.execute("PRAGMA foreign_keys = ON")
store = SqliteStore(store_connection, index={"embed": embeddings, "dims": 768})
store.setup()
# [작성] A의 visual_style에서 text를 읽으세요.
restored_style = ...
print("재연결 후:", restored_style)


In [ ]:
# [자가채점] 평가용 변수는 함수 안에서만 사용합니다. 모델을 추가 호출하지 않습니다.
def grade_question_12():
    """지문의 요구사항과 작성 결과를 비교합니다."""
    assert restored_style == updated_style, "같은 파일·namespace·key로 변경된 기억을 읽으세요."
    print("12번 자가평가 통과")

grade_question_12()


## 13. 이미지와 저장 범위를 설명합니다

**배경:** 다른 제작자가 이어서 사용할 수 있도록 동작을 설명합니다.

**작성:** 서술형 답안

**요구사항:**
- 두 브랜드 결과에서 화풍·색·구도와 공통 사물이 어떻게 반영됐는지 각각 적으세요. 반영되지 않은 조건도 기록합니다.
- user_id·namespace·key·thread_id의 역할을 이 과제의 예로 설명하세요.
- Store·Checkpointer·PNG 파일에 각각 무엇이 저장되는지, 새 제작에서 화풍은 공유되는데 이전 요청은 섞이지 않는 이유를 설명하세요.
- 11번에서 화풍만 바꿨을 때 나머지 제작 참고와 어울렸는지도 적으세요. 이미지 차이만으로 기억 효과를 단정하지 않습니다.

<details><summary>힌트</summary>

```text
조회한 기억 → 생성 프롬프트 → 실제 이미지 순서로 대조합니다.
```

</details>

**확인:** 정답 노트북의 모범 설명과 비교하되, 같은 의미를 자신의 말로 설명해도 됩니다.


*(여기에 판단의 이유와 확인한 결과를 작성하세요.)*


## 마무리합니다

- [ ] 두 브랜드의 요청은 같고 각자의 기억을 사용했습니다.
- [ ] 새 제작·화풍 변경·재연결 결과를 확인했습니다.
- [ ] 실제 이미지와 조회 선호를 대조한 서술을 남겼습니다.

연결 종료 셀을 실행하세요. 닫힌 연결로 그래프를 다시 실행하려면 준비 셀에서 연결을 열고 다시 컴파일합니다.


In [ ]:
# [제공 코드] 파일은 남기고 열린 연결을 닫습니다.
store_connection.close()
connection.close()
